# SENTINEL — LLM GRPO Training Pipeline
### OpenEnv Hackathon 2026 | Meta PyTorch

This notebook demonstrates the **Group Relative Policy Optimization (GRPO)** training pipeline for the SENTINEL environment using **Unsloth** and **HF TRL**.

**Objective:** Train a Llama-3.2-1B model to perform multi-step root cause analysis and remediation on cascading microservice failures.

## 1. Setup Environment & Dependencies

In [ ]:
import subprocess, sys, os
if not os.path.exists('sentinel'):
    subprocess.run(['git', 'clone', 'https://github.com/SayantikaLaskar/sentinel.git'], check=False)
    os.chdir('sentinel')
elif os.path.basename(os.getcwd()) != 'sentinel':
    os.chdir('sentinel')

# Install core requirements
!pip install -q -r requirements.txt

# Install LLM specialized libraries
!pip install -q "unsloth[colab-new] @ git+https://github.com/unslothai/unsloth.git"
!pip install -q --no-deps "trl<0.12.0" peft accelerate bitsandbytes

print('Environment Ready.')

## 2. Initialize GRPO Training Loop

We use **Unsloth** for 4-bit quantization and **TRL's GRPOTrainer** for the RLVR (Reinforcement Learning from Verifiable Rewards) signal.

In [ ]:
import torch
from sentinel.training.pipeline import build_grpo_trainer, run_training_loop, TrainingConfig
from sentinel.env import Sentinel_Env

if not torch.cuda.is_available():
    raise RuntimeError('This notebook requires a GPU. Please change the runtime type.')

# 1. Initialize Environment
env = Sentinel_Env(render_mode='human')
reward_fn = env.reward_function

# 2. Configure Training
config = TrainingConfig(
    agent='holmes',
    model_name='unsloth/Llama-3.2-1B-Instruct-bnb-4bit',
    max_steps=50,  # Short run for demonstration
    batch_size=1,
    gradient_accumulation_steps=4
)

# 3. Build Hybrid Trainer
print('Loading model and building GRPOTrainer...')
trainer, llm_agent = build_grpo_trainer(agent='holmes', env=env, config=config)

# 4. Start GRPO Training Loop
if trainer:
    print('\nStarting GRPO Training...')
    run_training_loop(
        env=env,
        trainer=trainer,
        llm_agent=llm_agent,
        config=config,
        reward_fn=reward_fn
    )
else:
    print('\nGRPOTrainer could not be built (likely due to VRAM limits).')

## 3. Visualize Results

In [ ]:
import matplotlib.pyplot as plt
import pandas as pd
import json

if os.path.exists('training_log.jsonl'):
    records = []
    with open('training_log.jsonl', 'r') as f:
        for line in f:
            if line.strip().startswith('{'):
                records.append(json.loads(line))
    
    if records:
        df = pd.DataFrame(records)
        plt.figure(figsize=(10, 5))
        plt.plot(df['episode'], df['total_reward'], label='Reward')
        plt.xlabel('Episode')
        plt.ylabel('Total Reward')
        plt.title('GRPO Training Progress')
        plt.grid(True)
        plt.show()
    else:
        print('No records found in log.')
else:
    print('training_log.jsonl not found.')